In [1]:
"""
PASSO 7 — TODIM + Métricas de Generalização.

Replica _IM_TODIM.R:
    1. TODIM: Lê matrizes {MÉTRICA}_{Sample}_{Período}.csv do Industry_Alg
       - Normaliza (maxmin para benefício, inverso para MDD e ART)
       - Aplica função de valor prospectiva (α=β=0.88, λ=2.25)
       - Ranqueia os 13 algoritmos por setor
    2. Métricas de Generalização: Compara rankings entre cenários
       - PR@k, RR@k, F1@k, CR@k, MRR, MAP, NDCG@k

Saída:
    - Ranking_TODIM_{cenário}.csv: ranking completo por setor
    - Generalization_Metrics.csv: tabela com todas as métricas

Uso:
    python 07_todim_metrics.py
"""

from pathlib import Path
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# ===================== CONFIGURAÇÃO =====================
BASE_DIR = Path(r"C:\Users\Rodrigo\Desktop\Replicando para B3_2\B3ICS")
SEC_NAMES = BASE_DIR / ".NewB3_pruned.csv"

ALGORITHMS = ["LR", "SVM", "NB", "CART", "RF", "XGB",
              "MLP", "DBN", "SAE", "RNN", "LSTM", "GRU"]

METRICS = ["WR", "ARR", "ASR", "MDD", "CAR", "SOR", "MSR", "ART"]

# Indicadores de custo (quanto MENOR, melhor): MDD e ART
COST_INDICATORS = ["MDD", "ART"]

# Pesos q dos indicadores (Seção 4.3 do paper)
# Ordem: WR, ARR, ASR, MDD, CAR, SOR, MSR, ART
Q_WEIGHTS = [1, 3, 4, 5, 6, 7, 2, 2]

# Parâmetros da função de valor prospectiva (Tversky & Kahneman)
ALPHA = 0.88   # ganho
BETA = 0.88    # perda
LAMBDA = 2.25  # aversão a perdas

# Cenários
SCENARIOS = {
    "In_InS":   {"Sample": "In",  "Period": "InS"},
    "In_OutS":  {"Sample": "In",  "Period": "OutS"},
    "Out_InS":  {"Sample": "Out", "Period": "InS"},
    "Out_OutS": {"Sample": "Out", "Period": "OutS"},
}

K_VALUES = [1, 3, 5]
# ========================================================


# ============================================================
#                    TODIM
# ============================================================

def maxmin_norm(x):
    """Normalização max-min para indicadores de benefício."""
    mn = np.nanmin(x)
    mx = np.nanmax(x)
    if mx == mn:
        return np.zeros_like(x)
    return (x - mn) / (mx - mn)


def cost_norm(x):
    """Normalização inversa para indicadores de custo (MDD, ART)."""
    mn = np.nanmin(x)
    mx = np.nanmax(x)
    if mx == mn:
        return np.zeros_like(x)
    return (mx - x) / (mx - mn)


def prospect_value(diff, w, alpha=ALPHA, beta=BETA, lam=LAMBDA):
    """
    Função de valor prospectiva (Eq. 13 do paper).
    diff = n_ij - n_kj (diferença entre dois modelos no critério j)
    w = peso normalizado do critério j
    """
    if diff > 0:
        return (w * diff) ** alpha
    elif diff == 0:
        return 0.0
    else:
        return -LAMBDA * ((-diff) / w) ** BETA


def load_industry_matrices(base_dir, sample, period):
    """
    Carrega as 8 matrizes de métricas e monta a matriz de decisão
    por setor. Retorna dict: {setor: DataFrame (n_algos × 8 métricas)}.
    """
    # Ler setores disponíveis
    codes_df = pd.read_csv(base_dir / ".NewB3_pruned.csv",
                           dtype=str, encoding="utf-8-sig")
    codes_df.columns = [c.strip() for c in codes_df.columns]
    industries = sorted(codes_df["industry"].str.strip().unique())

    # Carregar cada métrica
    metric_data = {}
    for metric in METRICS:
        filepath = base_dir / f"{metric}_{sample}_{period}.csv"
        if filepath.exists():
            df = pd.read_csv(filepath, index_col=0, encoding="utf-8-sig")
            metric_data[metric] = df
        else:
            print(f"  [AVISO] Arquivo não encontrado: {filepath.name}")

    if not metric_data:
        return None, industries

    # Montar matrizes por setor
    sector_matrices = {}
    for ind in industries:
        rows = []
        for metric in METRICS:
            if metric in metric_data and ind in metric_data[metric].columns:
                col = metric_data[metric][ind]
                rows.append(col)
            else:
                rows.append(pd.Series(np.nan, index=metric_data[METRICS[0]].index
                                      if METRICS[0] in metric_data else ALGORITHMS))
        # Transpor: linhas=modelos, colunas=métricas
        mat = pd.DataFrame(rows, index=METRICS).T
        sector_matrices[ind] = mat

    return sector_matrices, industries


def todim_rank_sector(decision_matrix):
    """
    Aplica TODIM a uma matriz de decisão de um setor.
    decision_matrix: DataFrame com linhas=[Index, BAH, LR, ...], colunas=métricas.
    Retorna ranking apenas dos algoritmos (sem Index e BAH).
    """
    # Remover Index e BAH
    algo_names = [a for a in decision_matrix.index if a not in ["Index", "BAH"]]
    mat = decision_matrix.loc[algo_names].copy()

    # Tratar NaN e Inf
    mat = mat.replace([np.inf, -np.inf], np.nan)
    global_mean = np.nanmean(mat.values)
    global_std = np.nanstd(mat.values)
    if global_std == 0:
        global_std = 1.0
    for col in mat.columns:
        mask = mat[col].isna()
        if mask.any():
            mat.loc[mask, col] = np.random.normal(global_mean, global_std, mask.sum())

    # Step 1: Normalização
    norm_mat = mat.copy()
    for col in METRICS:
        if col in norm_mat.columns:
            if col in COST_INDICATORS:
                norm_mat[col] = cost_norm(norm_mat[col].values)
            else:
                norm_mat[col] = maxmin_norm(norm_mat[col].values)

    # Step 2: Pesos normalizados
    q = np.array(Q_WEIGHTS, dtype=float)
    w = q / q.max()

    N = len(algo_names)
    M = len(METRICS)

    # Step 3-5: Calcular vantagem total de cada modelo
    phi_total = np.zeros(N)

    for j in range(N):
        ressum = 0.0
        for m in range(M):
            col = METRICS[m]
            if col not in norm_mat.columns:
                continue
            for k in range(N):
                diff = norm_mat.iloc[j][col] - norm_mat.iloc[k][col]
                pv = prospect_value(diff, w[m])
                ressum += pv
        phi_total[j] = ressum

    # Step 6: Normalização final max-min
    if phi_total.max() == phi_total.min():
        xi = np.zeros(N)
    else:
        xi = (phi_total - phi_total.min()) / (phi_total.max() - phi_total.min())

    # Step 7: Ranking (decrescente)
    ranking = pd.Series(xi, index=algo_names)
    ranking = ranking.sort_values(ascending=False)

    return ranking


def run_todim(base_dir, sample, period, scenario_label):
    """
    Executa TODIM para todos os setores de um cenário.
    Retorna DataFrame com ranking por setor.
    """
    sector_matrices, industries = load_industry_matrices(base_dir, sample, period)

    if sector_matrices is None:
        print(f"  [ERRO] Sem dados para cenário {scenario_label}")
        return None

    rankings = {}
    for ind in industries:
        if ind in sector_matrices:
            ranking = todim_rank_sector(sector_matrices[ind])
            rankings[ind] = ranking.index.tolist()  # lista ordenada de algoritmos

    # Montar DataFrame: colunas=setores, linhas=ranking (1º, 2º, ...)
    max_len = max(len(v) for v in rankings.values()) if rankings else 0
    rank_df = pd.DataFrame(index=range(1, max_len + 1))
    rank_df.index.name = "Rank"
    for ind in industries:
        if ind in rankings:
            rank_df[ind] = rankings[ind] + [None] * (max_len - len(rankings[ind]))

    # Salvar
    outfile = base_dir / f"Ranking_TODIM_{scenario_label}.csv"
    rank_df.to_csv(outfile, encoding="utf-8-sig")
    print(f"  Ranking salvo: {outfile.name}")

    return rank_df


# ============================================================
#              MÉTRICAS DE GENERALIZAÇÃO
# ============================================================

def get_topk(rank_df, sector, k):
    """Retorna set dos top-k algoritmos de um setor."""
    if sector not in rank_df.columns:
        return set()
    vals = rank_df[sector].dropna().tolist()[:k]
    return set(vals)


def get_topk_list(rank_df, sector, k):
    """Retorna lista ordenada dos top-k algoritmos."""
    if sector not in rank_df.columns:
        return []
    return rank_df[sector].dropna().tolist()[:k]


def calc_pr_rr_f1(baseline, target, sectors, k):
    """PR@k, RR@k, F1@k (Eqs. 17-19 do paper)."""
    base_union = set()
    target_union = set()
    for s in sectors:
        base_union |= get_topk(baseline, s, k)
        target_union |= get_topk(target, s, k)

    intersect = len(base_union & target_union)
    pr = intersect / len(base_union) if len(base_union) > 0 else 0
    rr = intersect / len(target_union) if len(target_union) > 0 else 0
    f1 = 2 * pr * rr / (pr + rr) if (pr + rr) > 0 else 0
    return pr, rr, f1


def calc_cr(target, sectors, k, total_algos):
    """CR@k (Eq. 20 do paper)."""
    all_recommended = set()
    for s in sectors:
        all_recommended |= get_topk(target, s, k)
    return len(all_recommended) / total_algos


def calc_mrr(baseline, target, sectors):
    """MRR (Eq. 21 do paper)."""
    rr_values = []
    for s in sectors:
        top1_target = get_topk_list(target, s, 1)
        if not top1_target:
            continue
        winner = top1_target[0]
        base_list = get_topk_list(baseline, s, len(ALGORITHMS))
        if winner in base_list:
            rank = base_list.index(winner) + 1
            rr_values.append(1.0 / rank)
        else:
            rr_values.append(0.0)
    return np.mean(rr_values) if rr_values else 0.0


def calc_map(baseline, target, sectors, total_algos):
    """MAP (Eq. 22 do paper)."""
    K = total_algos
    ap_values = []
    for s in sectors:
        base_list = get_topk_list(baseline, s, K)
        target_list = get_topk_list(target, s, K)
        if not base_list or not target_list:
            continue
        precision_sum = 0.0
        for i in range(1, K + 1):
            set_base = set(base_list[:i])
            set_target = set(target_list[:i])
            hits = len(set_base & set_target)
            precision_sum += hits / i
        ap_values.append(precision_sum / K)
    return np.mean(ap_values) if ap_values else 0.0


def calc_ndcg(baseline, target, sectors, k):
    """NDCG@k (Eqs. 23-25 do paper)."""
    ndcg_values = []
    for s in sectors:
        base_list = get_topk_list(baseline, s, len(ALGORITHMS))
        target_list = get_topk_list(target, s, k)
        if not base_list or not target_list:
            continue

        # Calcular ranks do target na lista baseline
        ranks = []
        for algo in target_list:
            if algo in base_list:
                ranks.append(base_list.index(algo) + 1)
            else:
                ranks.append(len(ALGORITHMS) + 1)

        # Correlação entre ranking ideal (1,2,...,k) e ranking observado
        if k == 1:
            rel = 1.0 if ranks[0] == 1 else 0.0
            ndcg_values.append(rel)
        else:
            from scipy.stats import spearmanr
            ideal = list(range(1, k + 1))
            corr, _ = spearmanr(ideal, ranks[:k])
            if np.isnan(corr):
                corr = 0.0

            # DCG / IDCG
            dcg = 0.0
            idcg = 0.0
            # Normalizar correlação para [0,1]
            rel_norm = maxmin_norm(np.array([corr]))[0] if not np.isnan(corr) else 0.0

            for i in range(k):
                idcg += (2 ** 1 - 1) / np.log2(i + 2)
                # Usar relevância baseada na correlação cumulativa
                sub_ranks = ranks[:i + 1]
                sub_ideal = list(range(1, i + 2))
                if len(sub_ranks) > 1:
                    c, _ = spearmanr(sub_ideal, sub_ranks)
                    if np.isnan(c):
                        c = 0.0
                else:
                    c = 1.0 if sub_ranks[0] == 1 else 0.0
                dcg += (2 ** c - 1) / np.log2(i + 2)

            ndcg_values.append(dcg / idcg if idcg > 0 else 0.0)

    return np.mean(ndcg_values) if ndcg_values else 0.0


# ============================================================
#                    EXECUÇÃO PRINCIPAL
# ============================================================

def main():
    print("=" * 60)
    print("PASSO 7: TODIM + Métricas de Generalização")
    print("=" * 60)

    # Ler setores
    codes_df = pd.read_csv(SEC_NAMES, dtype=str, encoding="utf-8-sig")
    codes_df.columns = [c.strip() for c in codes_df.columns]
    industries = sorted(codes_df["industry"].str.strip().unique())

    print(f"Setores: {industries}")
    print(f"Algoritmos: {len(ALGORITHMS)}")
    print(f"Pesos q: {dict(zip(METRICS, Q_WEIGHTS))}\n")

    # --- 1. Executar TODIM para os 4 cenários ---
    print("--- TODIM Ranking ---")
    rankings = {}
    for scen_name, scen_params in SCENARIOS.items():
        print(f"\nCenário: {scen_name}")
        rank_df = run_todim(
            BASE_DIR,
            scen_params["Sample"],
            scen_params["Period"],
            scen_name,
        )
        if rank_df is not None:
            rankings[scen_name] = rank_df

    if "In_InS" not in rankings:
        print("\n[ERRO] Ranking do InS-TraD não gerado. Impossível calcular métricas.")
        return

    # Mostrar top-1 por setor no InS-TraD
    baseline = rankings["In_InS"]
    print(f"\n--- Modelo ótimo por setor (InS-TraD) ---")
    for ind in industries:
        if ind in baseline.columns:
            print(f"  {ind}: {baseline[ind].iloc[0]}")

    # --- 2. Métricas de Generalização ---
    print(f"\n--- Métricas de Generalização ---")
    target_scenarios = {k: v for k, v in rankings.items() if k != "In_InS"}

    results = []
    for scen_name, target in target_scenarios.items():
        sectors = [s for s in industries if s in baseline.columns and s in target.columns]
        total_algos = len(ALGORITHMS)

        # Métricas globais
        mrr = calc_mrr(baseline, target, sectors)
        map_val = calc_map(baseline, target, sectors, total_algos)

        print(f"\n  Cenário: {scen_name}")
        print(f"    MRR: {mrr:.4f} | MAP: {map_val:.4f}")

        for k in K_VALUES:
            pr, rr, f1 = calc_pr_rr_f1(baseline, target, sectors, k)
            cr = calc_cr(target, sectors, k, total_algos)
            ndcg = calc_ndcg(baseline, target, sectors, k)

            print(f"    K={k}: PR={pr:.3f} RR={rr:.3f} F1={f1:.3f} CR={cr:.3f} NDCG={ndcg:.3f}")

            results.append({
                "Scenario": scen_name,
                "K": k,
                "PR": round(pr, 4),
                "RR": round(rr, 4),
                "F1": round(f1, 4),
                "CR": round(cr, 4),
                "MRR": round(mrr, 4),
                "MAP": round(map_val, 4),
                "NDCG": round(ndcg, 4),
            })

    # Salvar
    if results:
        results_df = pd.DataFrame(results)
        outfile = BASE_DIR / "Generalization_Metrics.csv"
        results_df.to_csv(outfile, index=False, encoding="utf-8-sig")
        print(f"\n  Métricas salvas: {outfile.name}")

    print(f"\n{'='*60}")
    print("Concluído!")


if __name__ == "__main__":
    main()

PASSO 7: TODIM + Métricas de Generalização
Setores: ['BM', 'CC', 'CNC', 'COM', 'FIN', 'GCS', 'OGB', 'UTI']
Algoritmos: 12
Pesos q: {'WR': 1, 'ARR': 3, 'ASR': 4, 'MDD': 5, 'CAR': 6, 'SOR': 7, 'MSR': 2, 'ART': 2}

--- TODIM Ranking ---

Cenário: In_InS
  Ranking salvo: Ranking_TODIM_In_InS.csv

Cenário: In_OutS
  Ranking salvo: Ranking_TODIM_In_OutS.csv

Cenário: Out_InS
  Ranking salvo: Ranking_TODIM_Out_InS.csv

Cenário: Out_OutS
  Ranking salvo: Ranking_TODIM_Out_OutS.csv

--- Modelo ótimo por setor (InS-TraD) ---
  BM: RF
  CC: SVM
  CNC: NB
  COM: RNN
  FIN: RF
  GCS: CART
  OGB: NB
  UTI: GRU

--- Métricas de Generalização ---

  Cenário: In_OutS
    MRR: 0.2939 | MAP: 0.6091
    K=1: PR=0.667 RR=0.800 F1=0.727 CR=0.417 NDCG=0.000
    K=3: PR=0.889 RR=0.800 F1=0.842 CR=0.833 NDCG=0.330
    K=5: PR=0.909 RR=0.909 F1=0.909 CR=0.917 NDCG=0.295

  Cenário: Out_InS
    MRR: 0.2418 | MAP: 0.5942
    K=1: PR=0.667 RR=0.800 F1=0.727 CR=0.417 NDCG=0.000
    K=3: PR=0.889 RR=1.000 F1=0.941 C

In [3]:
"""
Exibe tabela dos modelos vencedores (top-1 TODIM) por setor e por cenário.

Lê os arquivos Ranking_TODIM_{Sample}_{Period}.csv e monta uma tabela
consolidada com o modelo ótimo de cada setor em cada cenário.

Saída:
    - Tabela_Modelos_Vencedores.csv
    - Tabela impressa no terminal

Uso:
    python tabela_modelos_vencedores.py
"""

from pathlib import Path
import pandas as pd

# ===================== CONFIGURAÇÃO =====================
BASE_DIR = Path(r"C:\Users\Rodrigo\Desktop\Replicando para B3_2\B3ICS")

SCENARIOS = {
    "InS-TraD":  "Ranking_TODIM_In_InS.csv",
    "InS-TesD":  "Ranking_TODIM_In_OutS.csv",
    "OutS-TraD": "Ranking_TODIM_Out_InS.csv",
    "OutS-TesD": "Ranking_TODIM_Out_OutS.csv",
}
# ========================================================


def main():
    results = {}

    for scen_name, filename in SCENARIOS.items():
        filepath = BASE_DIR / filename
        if not filepath.exists():
            print(f"[AVISO] {filename} não encontrado.")
            continue

        rank_df = pd.read_csv(filepath, index_col=0, encoding="utf-8-sig")

        # Top-1 de cada setor (primeira linha do ranking)
        top1 = {}
        for col in rank_df.columns:
            vals = rank_df[col].dropna()
            if len(vals) > 0:
                top1[col] = vals.iloc[0]
            else:
                top1[col] = "—"

        results[scen_name] = top1

    if not results:
        print("[ERRO] Nenhum arquivo de ranking encontrado.")
        return

    # Montar tabela consolidada
    table = pd.DataFrame(results)

    # Ordenar setores
    table = table.sort_index()

    # Reordenar colunas
    col_order = ["InS-TraD", "InS-TesD", "OutS-TraD", "OutS-TesD"]
    col_order = [c for c in col_order if c in table.columns]
    table = table[col_order]

    # Imprimir
    print("\n" + "=" * 70)
    print("MODELOS VENCEDORES (TOP-1 TODIM) POR SETOR E CENÁRIO")
    print("=" * 70)

    # Formatação alinhada
    header = f"{'Setor':>6}"
    for col in col_order:
        header += f"  {col:>10}"
    print(header)
    print("-" * len(header))

    for setor in table.index:
        row = f"{setor:>6}"
        for col in col_order:
            val = table.loc[setor, col] if col in table.columns else "—"
            row += f"  {val:>10}"
        print(row)

    print("-" * len(header))

    # Análise de consistência
    if "InS-TraD" in table.columns:
        print(f"\n{'='*70}")
        print("ANÁLISE DE GENERALIZAÇÃO (InS-TraD como referência)")
        print("=" * 70)

        ref = table["InS-TraD"]
        for scen in col_order[1:]:
            if scen not in table.columns:
                continue
            comp = table[scen]
            matches = (ref == comp).sum()
            total = len(ref)
            print(f"  {scen:>10}: {matches}/{total} setores coincidem "
                  f"({matches/total*100:.0f}%)")

    # Análise de frequência dos modelos
    print(f"\n{'='*70}")
    print("FREQUÊNCIA DOS MODELOS COMO TOP-1")
    print("=" * 70)

    all_models = []
    for col in col_order:
        if col in table.columns:
            all_models.extend(table[col].tolist())

    from collections import Counter
    freq = Counter(all_models)
    for model, count in freq.most_common():
        if model == "—": continue
        total_slots = len(table) * len(col_order)
        print(f"  {model:>6}: {count} vezes "
              f"({count/total_slots*100:.0f}% dos {total_slots} slots)")

    # Salvar
    outfile = BASE_DIR / "Tabela_Modelos_Vencedores.csv"
    table.to_csv(outfile, encoding="utf-8-sig")
    print(f"\nTabela salva: {outfile.name}")

    # Também salvar Best_Model_Per_Stock se existir
    bp_file = BASE_DIR / "Best_Model_Per_Stock.csv"
    if bp_file.exists():
        bp_df = pd.read_csv(bp_file, encoding="utf-8-sig")
        print(f"\n{'='*70}")
        print("MODELOS VENCEDORES POR AÇÃO INDIVIDUAL")
        print("=" * 70)

        freq_ind = Counter(bp_df["BestModel"].tolist())
        for model, count in freq_ind.most_common():
            print(f"  {model:>6}: {count} ações "
                  f"({count/len(bp_df)*100:.0f}%)")


if __name__ == "__main__":
    main()


MODELOS VENCEDORES (TOP-1 TODIM) POR SETOR E CENÁRIO
 Setor    InS-TraD    InS-TesD   OutS-TraD   OutS-TesD
------------------------------------------------------
    BM          RF         SVM          LR          NB
    CC         SVM          LR          LR         XGB
   CNC          NB        CART         RNN         GRU
   COM         RNN          NB        CART         XGB
   FIN          RF        CART         SVM         RNN
   GCS        CART         SVM          RF         DBN
   OGB          NB         GRU        CART         XGB
   UTI         GRU         SVM          RF         GRU
------------------------------------------------------

ANÁLISE DE GENERALIZAÇÃO (InS-TraD como referência)
    InS-TesD: 0/8 setores coincidem (0%)
   OutS-TraD: 0/8 setores coincidem (0%)
   OutS-TesD: 1/8 setores coincidem (12%)

FREQUÊNCIA DOS MODELOS COMO TOP-1
     SVM: 5 vezes (16% dos 32 slots)
    CART: 5 vezes (16% dos 32 slots)
      RF: 4 vezes (12% dos 32 slots)
      NB: 4 vezes 